# Table Data Accumulation and Creation

In [1]:
import requests
import pandas as pd
from io import StringIO

In [2]:
# wikipedia scraping for yearly evaluations of NFL franchises

url = "https://en.wikipedia.org/wiki/Forbes_list_of_the_most_valuable_NFL_teams"
headers = {"User-Agent": "Mozilla/5.0 (research script)"}
resp = requests.get(url, headers=headers, timeout=15)
resp.raise_for_status()

In [3]:
# Creates a list of tables from the HTML scrape
tables = pd.read_html(StringIO(resp.text))

In [4]:
# Loop logic that runs through each table in the list and returns the relevant table based on specified conditions
running_best_count = 0.0
running_best_table = None
historical_table = None

for t in tables:
    col_names_list = list(t.columns)
    column_list = [] # Empty list that will contain the name of each valid column
    for c in col_names_list:
        c = str(c)
        if c.isdigit() and len(c) == 4:
                column_list.append(c)
        fraction_count = len(column_list)/len(col_names_list)
        if fraction_count > running_best_count:
            running_best_count = fraction_count
            running_best_table = t

if running_best_table is None:
      print("No historical table found")
      raise TypeError 

historical_table = pd.DataFrame(running_best_table)

In [5]:
# Melting the historical fact table from one row per team to one row per team per year
evaluations = historical_table.melt(id_vars=["Team"], var_name="Year", value_name="Evaluation").sort_values(by=["Team", "Year"]).reset_index(drop=True)

In [6]:
# Changing year column dtype to int and making all column names lowercase and returning the df info and df
evaluations["Year"] = evaluations["Year"].astype(int)
evaluations.columns = evaluations.columns.str.lower()

evaluations.info()

evaluations

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 384 entries, 0 to 383
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   team        384 non-null    object
 1   year        384 non-null    int64 
 2   evaluation  384 non-null    int64 
dtypes: int64(2), object(1)
memory usage: 9.1+ KB


,team,year,evaluation
0,Arizona Cardinals,2012,961
1,Arizona Cardinals,2013,1000
2,Arizona Cardinals,2014,1500
3,Arizona Cardinals,2015,2000
4,Arizona Cardinals,2016,2200
...,...,...,...
379,Washington Commanders,2019,3500
380,Washington Commanders,2020,4200
381,Washington Commanders,2021,5600
382,Washington Commanders,2022,6050


In [7]:
# Makes a unique array of all teams and creates a new dimension table with team id, name, and league 
all_teams = evaluations["team"].unique()

teams = pd.DataFrame({"team_id":range(len(all_teams)), "team":all_teams, "league":"NFL"})

teams

,team_id,team,league
0,0,Arizona Cardinals,NFL
1,1,Atlanta Falcons,NFL
2,2,Baltimore Ravens,NFL
3,3,Buffalo Bills,NFL
4,4,Carolina Panthers,NFL
5,5,Chicago Bears,NFL
6,6,Cincinnati Bengals,NFL
7,7,Cleveland Browns,NFL
8,8,Dallas Cowboys,NFL
9,9,Denver Broncos,NFL


In [8]:
# Changes the representation of each team in the evaluations table to its team_id as a fact table to reference the dimensions
valuations = evaluations.merge(teams, how="inner", on= "team")[["team_id", "year", "evaluation"]]

valuations

,team_id,year,evaluation
0,0,2012,961
1,0,2013,1000
2,0,2014,1500
3,0,2015,2000
4,0,2016,2200
...,...,...,...
379,31,2019,3500
380,31,2020,4200
381,31,2021,5600
382,31,2022,6050


In [9]:
# Creates a dictionary for every team and its current city to match against the evaluations table to attach a city for each team/year row
current_city = {
    "Arizona Cardinals": "Phoenix",
    "Atlanta Falcons": "Atlanta",
    "Baltimore Ravens": "Baltimore",
    "Buffalo Bills": "Buffalo",
    "Carolina Panthers": "Charlotte",
    "Chicago Bears": "Chicago",
    "Cincinnati Bengals": "Cincinnati",
    "Cleveland Browns": "Cleveland",
    "Dallas Cowboys": "Dallas",
    "Denver Broncos": "Denver",
    "Detroit Lions": "Detroit",
    "Green Bay Packers": "Green Bay",
    "Houston Texans": "Houston",
    "Indianapolis Colts": "Indianapolis",
    "Jacksonville Jaguars": "Jacksonville",
    "Kansas City Chiefs": "Kansas City",
    "Las Vegas Raiders": "Las Vegas",
    "Los Angeles Chargers": "Los Angeles",
    "Los Angeles Rams": "Los Angeles",
    "Miami Dolphins": "Miami",
    "Minnesota Vikings": "Minneapolis",
    "New England Patriots": "Boston",
    "New Orleans Saints": "New Orleans",
    "New York Giants": "New York",
    "New York Jets": "New York",
    "Philadelphia Eagles": "Philadelphia",
    "Pittsburgh Steelers": "Pittsburgh",
    "San Francisco 49ers": "San Francisco",
    "Seattle Seahawks": "Seattle",
    "Tampa Bay Buccaneers": "Tampa",
    "Tennessee Titans": "Nashville",
    "Washington Commanders": "Washington",
}

In [10]:
# Relocation exceptions: team -> (year it moved, city before the move)
relocations = {
    "Los Angeles Rams":     {"cutover_year": 2016, "old_city": "St. Louis"},
    "Los Angeles Chargers": {"cutover_year": 2017, "old_city": "San Diego"},
    "Las Vegas Raiders":    {"cutover_year": 2020, "old_city": "Oakland"},
}

In [11]:
# Step 1: give every row its team's current city as the default using the map method and the current city dictionary
evaluations["city"] = evaluations["team"].map(current_city)

In [12]:
# Step 2: for the 3 relocated teams, overwrite the years before the move
for team, info in relocations.items():
    mask = (evaluations["team"] == team) & (evaluations["year"] < info["cutover_year"])
    evaluations.loc[mask, "city"] = info["old_city"]

In [13]:
# Sanity check to ensure proper relocations were calcualted and inputed
evaluations[evaluations["team"].isin(relocations)].groupby(["team", "city"])["year"].agg(["min", "max", "count"])

min   max  count
team                 city                          
Las Vegas Raiders    Las Vegas    2020  2023      4
                     Oakland      2012  2019      8
Los Angeles Chargers Los Angeles  2017  2023      7
                     San Diego    2012  2016      5
Los Angeles Rams     Los Angeles  2016  2023      8
                     St. Louis    2012  2015      4

In [14]:
# overwrites the old evaluations variable's job with a properly-named valuations table
valuations = evaluations.merge(teams, how="inner", on="team")[["team_id", "year", "evaluation", "city"]]

In [15]:
# imports the sqlite library, creates and connects to a database, and uploads the two exisiting tables into the database and commits them
import sqlite3

conn = sqlite3.connect(r"c:\Users\lpeco\OneDrive\Desktop\Development\Projects\Full_Pipeline\sports_valuation_project\SQL\sports_valuation.db")

teams.to_sql("teams", conn, if_exists="replace", index=False)
valuations.to_sql("valuations", conn, if_exists="replace", index=False)

conn.commit()

# Gathering media market data

In [16]:
# Scrapes the wikipedia page containing each DMA and its Nielsen ranking

url = "https://en.wikipedia.org/wiki/List_of_United_States_television_markets"
headers = {"User-Agent": "Mozilla/5.0 (research script)"}
resp = requests.get(url, headers=headers, timeout=15)
resp.raise_for_status()

In [17]:
import re

# Uses regular expression to parse the raw wikitext of each template link and pull the market name and rank
pattern = re.compile(r'\[\[Template:[^|\]]*\|([^\]]+)\]\]\s*\(#(\d+)\)')

# Scans resp.text for every match and returns a list of (market_name, rank) tuples, one per market found on the page, in page order
matches = pattern.findall(resp.text)

In [18]:
# Creates a dataframe from the list of tuples in "matches" variable and assigns columns 
pd.set_option('display.max.rows', 20)

media_markets = pd.DataFrame(matches, columns=["market_name", "media_market_rank"])

# Type casts the "media_market_rank" from its original dtype to an "int" dtype
media_markets["media_market_rank"] = media_markets["media_market_rank"].astype(int)

media_markets

,market_name,media_market_rank
0,New York,1
1,Los Angeles,2
2,Chicago,3
3,Dallas–Fort Worth,4
4,Philadelphia,5
...,...,...
205,Presque Isle,206
206,Juneau,207
207,Alpena,208
208,North Platte,209


In [19]:
# Loop logic that parses through every row in the media_markets dataframe and standardizes all of the city names to match the key format found in the valuations table

media_markets_list = media_markets["market_name"]
ranking = media_markets["media_market_rank"]

market_list = [m.split("–")[0] for m in media_markets_list]
area_list = [c.split("(")[0] for c in market_list]
city_list = [a.split(",")[0] for a in area_list]
final_market_list = [fm.strip() for fm in city_list]

media_markets = pd.DataFrame({"market_name": final_market_list, "media_market_rank":ranking})

san_fran_ranking = media_markets[media_markets["market_name"] == "San Francisco"]["media_market_rank"].to_list()
oakland = pd.DataFrame({"market_name": "Oakland", "media_market_rank":san_fran_ranking})

media_markets = pd.concat([media_markets, oakland], ignore_index=True)
media_markets.sort_values(by='media_market_rank', ascending=True)

,market_name,media_market_rank
0,New York,1
1,Los Angeles,2
2,Chicago,3
3,Dallas,4
4,Philadelphia,5
...,...,...
205,Presque Isle,206
206,Juneau,207
207,Alpena,208
208,North Platte,209


In [20]:
# Merges the two dataframes to remove any unmatched cities and their rankings from the media_markets table

media_markets = media_markets.merge(valuations[["city"]].drop_duplicates(), how='inner', left_on="market_name", right_on="city")[["market_name", "media_market_rank"]]

media_markets

,market_name,media_market_rank
0,New York,1
1,Los Angeles,2
2,Chicago,3
3,Dallas,4
4,Philadelphia,5
...,...,...
28,Jacksonville,41
29,New Orleans,50
30,Buffalo,55
31,Green Bay,68


In [21]:
# Commits the table to the SQLite database and completes the schema

media_markets.to_sql("media_markets", conn, if_exists="replace", index=False)
conn.commit()